In [12]:
import numpy as np

import data.breathe_data as bd
import data.helpers as dh
import models.builders as mb
from plotly.subplots import make_subplots
from plotly import graph_objects as go
import inference.helpers as ih

import pandas as pd

In [8]:
df = bd.load_meas_from_excel("BR_O2_FEV1_FEF2575_conservative_smoothing_with_idx")

INFO:root:* Checking for same day measurements *


In [28]:
df_long_fev = bd.load_meas_from_excel(
    "long_model/laplace1.7_30days_fev1",
    str_cols_to_arrays=["AR", "HFEV1"],
).rename(columns={"AR": "AR (fev1)", "HFEV1": "HFEV1 (fev1)"})
df_long_fev_fef = bd.load_meas_from_excel(
    "long_model/laplace1.7_30days_fev1_fef2575",
    str_cols_to_arrays=["AR", "HFEV1"],
).rename(columns={"AR": "AR (fev1 fef2575)", "HFEV1": "HFEV1 (fev1 fef2575)"})

INFO:root:* Checking for same day measurements *
INFO:root:* Checking for same day measurements *


In [23]:
ecfev1_noise_model_suffix = "_std_add_mult_ecfev1"
ar_change_cpt_suffix = "_shape_factor_single_laplace_1.7"

(
    _,
    _,
    HFEV1,
    uFEV1,
    ecFEV1,
    AR,
    _,
    S,
) = mb.fev1_fef2575_long_model_noise_shared_healthy_vars_and_temporal_ar(
    175,
    20,
    "Male",
    ar_change_cpt_suffix=ar_change_cpt_suffix,
    ecfev1_noise_model_suffix=ecfev1_noise_model_suffix,
    fef2575_cpt_suffix="",
    light=False,
)

In [ ]:
df.columns
cols_2_keep = ["ID", "Date Recorded", "ecFEV1", "ecFEF2575%ecFEV1"]
df_agg = df[cols_2_keep].merge(df_long_fev, on=["ID", "Date Recorded"], how="left")
df_agg = df_agg.merge(df_long_fev_fef, on=["ID", "Date Recorded", "Sequence length"], how="left")
df_agg.head()

,ID,Date Recorded,ecFEV1,ecFEF2575%ecFEV1,AR (fev1),HFEV1 (fev1),Sequence length,AR (fev1 fef2575),HFEV1 (fev1 fef2575)
0,101,2019-01-25,1.31,41.221374,"[0.00647990751, 0.0129244589, 0.0152976783, 0....","[0.0, 2.48265652e-225, 4.64471323e-141, 1.8114...",30.0,"[5.82898102e-36, 4.85711093e-34, 2.79983175e-3...","[0.0, 5.34856365e-270, 7.32922945e-186, 2.2953..."
1,101,2019-01-26,1.31,43.511450,"[0.00786464707, 0.0148425889, 0.0155174175, 0....","[0.0, 2.48265652e-225, 4.64471323e-141, 1.8114...",30.0,"[4.38145704e-37, 2.85427518e-35, 2.43486003e-3...","[0.0, 5.34856365e-270, 7.32922945e-186, 2.2953..."
2,101,2019-01-27,1.31,51.145038,"[0.00816496862, 0.0152566869, 0.0154417145, 0....","[0.0, 2.48265652e-225, 4.64471323e-141, 1.8114...",30.0,"[3.19270491e-37, 1.77351662e-35, 1.42383855e-3...","[0.0, 5.34856365e-270, 7.32922945e-186, 2.2953..."
3,101,2019-01-28,1.30,53.076923,"[0.00715339428, 0.0143292031, 0.0157270385, 0....","[0.0, 2.48265652e-225, 4.64471323e-141, 1.8114...",30.0,"[1.87169758e-37, 9.96542568e-36, 7.85080576e-3...","[0.0, 5.34856365e-270, 7.32922945e-186, 2.2953..."
4,101,2019-01-29,1.28,46.875000,"[0.00466535546, 0.0118155387, 0.0156528445, 0....","[0.0, 2.48265652e-225, 4.64471323e-141, 1.8114...",30.0,"[7.76273906e-39, 4.95609165e-37, 5.2496043e-35...","[0.0, 5.34856365e-270, 7.32922945e-186, 2.2953..."


In [14]:
df_for_ID = df_long_fev
p_HFEV1 = df_long_fev.HFEV1[0]

id = df.loc[0, "ID"]
layout = [
    [
        {"type": "scatter", "rowspan": 1, "colspan": 1},
        {"type": "scatter", "rowspan": 1, "colspan": 1},
    ],
    [None, {"type": "scatter", "rowspan": 1, "colspan": 1}],
    [None, {"type": "heatmap", "rowspan": 2, "colspan": 1}],
    [None, None],
]
fig = make_subplots(
    rows=np.shape(layout)[0],
    cols=np.shape(layout)[1],
    specs=layout,
    vertical_spacing=0.05,
    horizontal_spacing=0.2,
    shared_xaxes=True,
)

# Add HFEV1 posterior
ih.plot_histogram(fig, HFEV1, p_HFEV1, 0, HFEV1.b, 1, 1, annot=True)
fig.update_xaxes(title_text=HFEV1.name, title_standoff=0.4, row=1, col=1)
fig.update_yaxes(title_text="p", row=1, col=1)

# Add ecFEV1
fig.add_trace(
    go.Scatter(
        y=df["ecFEV1"],
        x=df["Date Recorded"],
        mode="lines+markers",
        marker=dict(size=4),
        line=dict(width=1),
    ),
    row=1,
    col=2,
)
fig.update_yaxes(
    title="ecFEV1 (L)",
    row=1,
    col=2,
    range=[np.nanmin(df.ecFEV1) * 0.98, np.nanmax(df.ecFEV1) * 1.02],
)

# Add ecFEF2575%ecFEV1
fig.add_trace(
    go.Scatter(
        y=df["ecFEF2575%ecFEV1"],
        x=df["Date Recorded"],
        mode="lines+markers",
        marker=dict(size=4),
        line=dict(width=1),
    ),
    row=2,
    col=2,
)
fig.update_yaxes(
    title="ecFEF2575<br>% ecFEV1",
    row=2,
    col=2,
    range=[
        np.nanmin(df["ecFEF2575%ecFEV1"]) * 0.98,
        np.nanmax(df["ecFEF2575%ecFEV1"]) * 1.02,
    ],
)

# Add heatmap with AR posteriors
df1 = pd.DataFrame(
    data=AR_given_M_and_D,
    columns=AR.get_bins_str(),
    index=df["Date Recorded"].apply(lambda date: date.strftime("%Y-%m-%d")),
)
colorscale = [
    [0, "white"],
    [0.01, "red"],
    [0.05, "yellow"],
    [0.1, "cyan"],
    [0.6, "blue"],
    [1, "black"],
]

fig.add_trace(
    go.Heatmap(z=df1.T, x=df1.index, y=df1.columns, coloraxis="coloraxis1"),
    row=3,
    col=2,
)
fig.update_yaxes(
    title_text=AR.name,
    tickmode="array",
    tickvals=np.linspace(AR.a, AR.b, 10),
    row=3,
    col=2,
)
fig.update_xaxes(
    title_text="Date (categorical)",
    row=3,
    col=2,
    nticks=10,
    type="category",
)

title = f"{id} - Results after fusing all P(M_h|D) {len(df)} entries"
fig.update_layout(
    font=dict(size=8),
    height=500,
    width=700,
    title=title,
    coloraxis1=dict(
        colorscale=colorscale,
        colorbar_x=1,
        colorbar_y=0.24,
        colorbar_thickness=10,
        colorbar_len=0.52,
    ),
    title_font_size=10,
    showlegend=False,
)

NameError: name 'AR_given_M_and_D' is not defined